# Messages
- Message object contains
    - `Role`: system/user etc.. Identifies message type 
    - `Content`: Content of the message like text, images, audio, document etc.
    - `Metadata`: Optional fields such as response info, message IDs, token usage etc.

- LangChain provides Standard Message types, that works with all the model providers

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

## Text Prompts
Use this when conversion history is not needed, for simple stragight forward standalone question and answers

In [2]:
model.invoke("Whhat is Loop Engineering in Agentic AI? explain in 100 words")

AIMessage(content='<think>\nOkay, let\'s tackle this question about Loop Engineering in Agentic AI. First, I need to make sure I understand what Agentic AI is. From what I know, agentic AI refers to AI systems that operate autonomously, making decisions and taking actions in an environment. They\'re often used in scenarios where the AI has a certain degree of autonomy, like in customer service bots or autonomous vehicles.\n\nNow, the term "Loop Engineering" isn\'t something I\'ve heard before. Let me break it down. In engineering, "loop" could refer to feedback loops, which are common in control systems. So maybe Loop Engineering here is about designing feedback loops for AI agents. That makes sense because feedback loops are crucial for learning and adaptation. \n\nAgentic AI would need to learn from their environment, adjust their actions based on outcomes, and optimize performance. So Loop Engineering might involve creating the mechanisms through which the AI agent continuously rece

## Message Prompts
We can pass list of messages to the model by providing the list of message objects

Message Types
- `System Message`: Tells the model how to behave and provides the context for interactions
    -  Act as an Experienced Data Scientist
    -  We can also provide like "Give all the responses in Caveman mode in very simple words in short instead of verbose responses"
- `Human Message`: User query to the LLM
- `AI Message`: Response generated by the LLM based on the query by the user or Tool call etc.
- `Tool Message`: Represents the output of the tool calls

In [5]:
from langchain.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("You are an expert Agentic AI Expert"),
    HumanMessage("Explain different Message Types in LangChain in short")
]
response = model.invoke(messages)
print(response.content)

<think>
Okay, so the user is asking me to explain the different message types in LangChain in a short manner. Let me start by recalling what LangChain is. LangChain is a framework for developing applications with language models, right? It helps in handling prompts, memory, agents, and more. Now, message types are probably related to how different kinds of messages are structured when interacting with language models.

First, I need to remember the specific message types. From what I've studied, LangChain uses a message abstraction where each message has a role and content. There are several types like SystemMessage, HumanMessage, AIMessage, etc. Maybe also FunctionMessage or ToolMessage?

Let me break them down. The SystemMessage is probably for setting the system's behavior, like instructions. The HumanMessage would be the user's input. The AIMessage is the response from the AI. Then there might be other messages for functions or tools used. Oh, right, there's also the FunctionMessag

In [ ]:
from langchain.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("You are an expert Agentic AI Expert, always provide the response in caveman mode cut verbose"),
    HumanMessage("Explain different Message Types in LangChain in short")
]
response = model.invoke(messages)
print(response.content)

<think>
Okay, the user wants a short explanation of different message types in LangChain. Let me recall what I know.

First, there's Prompt and PromptTemplate. They create input for models, like a simple string or a template with variables. Then there's Messages, which are objects like HumanMessage, AIMessage, SystemMessage. They structure the conversation. Chains are sequences of steps, maybe combining prompts and models. Agents use LLMs to decide actions, like using tools. Memories help retain context across interactions. Callbacks handle events during execution. Outputs are the results from chains or agents. Maybe there's also a way to handle errors or exceptions.

I need to make sure I cover each type briefly. Keep it simple, avoid jargon. Use short sentences. The user might be new to LangChain, so clarity is key. Let me list them out in bullet points and check if I missed anything important. Oh, and the user specified to explain in a caveman model, so very short and to the point. 

## Making Model understand how to Respond

### Default Model Response

In [9]:
# Another Example
from langchain.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    HumanMessage("10/2=?"),
]
response = model.invoke(messages)
print(response.content)

<think>
Okay, so I need to figure out what 10 divided by 2 is. Let me think. Hmm, division is like splitting a number into equal parts. So if I have 10 apples and I want to divide them into 2 groups, how many apples would each group have? Let me visualize this. If I split 10 into two equal parts, each part should be half of 10. Half of 10 is 5, right? Wait, does that make sense? Let me check another way. Multiplication is the inverse of division, so if I multiply 2 by some number to get 10, that number should be the answer. So 2 times what equals 10? Well, 2 times 5 is 10. Therefore, 10 divided by 2 must be 5. Let me confirm with another method. If I subtract 2 from 10 repeatedly until I reach 0, how many times do I subtract? 10 - 2 = 8, that's 1 time. 8 - 2 = 6, that's 2. 6 - 2 = 4, that's 3. 4 - 2 = 2, that's 4. 2 - 2 = 0, that's 5. So it took 5 subtractions, which means 10 divided by 2 is 5. All these methods are pointing to the same answer. I think that's solid. No contradictions h

### Tuned Model Response

In [7]:
# Another Example
from langchain.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("You are an expert Agentic AI Expert"),
    HumanMessage("What is 2+2=?"),
    AIMessage("2+2=4"),
    HumanMessage("4*2=?"),
    AIMessage("8"),
    HumanMessage("10/2=?")
]
response = model.invoke(messages)
print(response.content)

<think>
Okay, so the user asked "10/2=?" Let me think. First, division is just splitting a number into equal parts. Here, 10 divided by 2 means how many times 2 fits into 10. Well, 2 times 5 is 10, so the answer should be 5. Let me double-check. If you have 10 items and you split them into 2 groups, each group has 5. Yeah, that makes sense. I don't see any tricks here, it's straightforward. The user has been asking simple arithmetic questions, so they probably just want the correct answer quickly. No need for extra explanation unless they ask for it. So the answer is 5.
</think>

10/2=5


In [ ]:
# to check the usage metadata
response.usage_metadata

{'input_tokens': 13, 'output_tokens': 568, 'total_tokens': 581}

## Making model understand on how to use Tool i.e., we are creating initial conversion as one-shot, so model understand how to use it

In [11]:
from langchain.messages import AIMessage, ToolMessage

# After model makes tool call
# Creating the AIMessage manually
ai_message = AIMessage(
    content=[],
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"location": "Hyderabad"},
            "id": "call_456"
        }
    ]
)

# Execute tool and create result message
weather_result = "44 C"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_456"
)

# continue conversation
messages = [
    HumanMessage("WHat is the weather in Hyderabad?"),
    ai_message,
    tool_message,
]

response = model.invoke(messages)
print(response.content)

<think>
Okay, the user asked for the weather in Hyderabad. I called the get_weather function with Hyderabad as the location. The response came back as 44°C. Now I need to present this information clearly.

First, I should mention the temperature. 44 degrees Celsius is quite hot, so maybe add a note about that. People might want to know if it's a good idea to go out or if they should stay hydrated. Also, maybe mention the general conditions—like if it's sunny or if there's a chance of rain. But since the data only gives the temperature, I should stick to that unless I can infer something else. 

I should check if there's any additional information needed. The user might appreciate a suggestion based on the temperature. Maybe recommend staying indoors during the hottest part of the day or using sunscreen if they have to go out. Also, converting Celsius to Fahrenheit could be helpful for some users, but since the original answer is in Celsius, I'll keep it that way unless instructed other